# BASELINE ONLY — repository settings and seeds
This trains **baseline 0, baseline 1 and baseline 2 only**. No adaptive training.
Attach only the original image dataset. Select a GPU, enable Internet and run.
Hyperparameters and seeds come from configs/train.yaml in GitHub, using the
same Ultralytics version/defaults as adaptive training. No saved adaptive
results or checkpoint upload is required. It uses the original pretrained weights.
Use the same experiment configuration as the adaptive runs for a fair comparison.
The frozen main notebook and adaptive results are not changed or deleted.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/binhdoquoc/GRL_RDD26.git"
REPO_REF = ""  # latest default branch; or a specific commit
REPO_DIR = Path("/kaggle/working/GRL_RDD26")
PROJECT_DIR = REPO_DIR

TRAIN_CONFIG = PROJECT_DIR / "configs/train.yaml"
# Change experiment.seeds in the repository to [0] for one baseline.
RUN_TESTS = False
TORCH_INDEX_URL = "https://pypi.org/simple"

## 1. Fetch the GitHub project

In [ ]:
def sync_repository():
    global PROJECT_FILES
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"Existing path is not a Git checkout: {REPO_DIR}")
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    if origin.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError(f"Existing checkout belongs to a different repository: {origin}")
    dirty = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, text=True)
    if dirty.strip():
        raise RuntimeError("Tracked local edits would affect this run. Update code in GitHub and use a clean checkout. No files were discarded.\n" + dirty)
    # HEAD resolves the remote default branch, even from a detached local checkout.
    subprocess.run(["git", "fetch", "origin", REPO_REF or "HEAD"], cwd=REPO_DIR, check=True)
    revision = subprocess.check_output(["git", "rev-parse", "FETCH_HEAD"], cwd=REPO_DIR, text=True).strip()
    subprocess.run(["git", "checkout", "--detach", revision], cwd=REPO_DIR, check=True)
    if not (PROJECT_DIR / "train.py").is_file():
        raise FileNotFoundError("The downloaded revision lacks train.py at the repository root.")
    print("Project:", PROJECT_DIR)
    print("GitHub revision used:", revision)
    PROJECT_FILES = [str(p.relative_to(PROJECT_DIR))
                     for folder in ("grl_yolo", "configs", "scripts", "tests")
                     for p in (PROJECT_DIR / folder).rglob("*")
                     if p.is_file() and p.suffix in {".py", ".yaml"}]
    PROJECT_FILES += ["train.py", "requirements.txt", "pyproject.toml", "pytest.ini", "README.md"]
    return revision

sync_repository()

## 2. Set up Python and verify the GPU

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
import importlib
import grl_yolo.environment as training_environment
# Reload restored code if this notebook is imported into an existing session.
importlib.reload(training_environment)
print("Notebook kernel:", sys.version)
PYTHON, ENV = training_environment.prepare_environment(PROJECT_DIR, torch_index=TORCH_INDEX_URL)

from grl_yolo.process import run_logged
COMMAND_NUMBER = 0

def run_project(arguments):
    global COMMAND_NUMBER, PYTHON, ENV
    if str(arguments[0]) in {"train.py", "scripts/run_baseline.py", "scripts/run_baseline_seeds.py"}:
        sync_repository()
        importlib.reload(training_environment)
        PYTHON, ENV = training_environment.prepare_environment(PROJECT_DIR, torch_index=TORCH_INDEX_URL)
    COMMAND_NUMBER += 1
    filename = "train.log" if str(arguments[0]) == "train.py" else f"command-{COMMAND_NUMBER:02d}.log"
    return run_logged([str(PYTHON), *map(str, arguments)], cwd=PROJECT_DIR, env=ENV,
                      log_file=PROJECT_DIR / "logs" / filename)

runtime = subprocess.run(
    [str(PYTHON), "-c", "import sys; print('Training Python:', sys.version, flush=True); "
     "import torch, ultralytics; print(torch.__version__, ultralytics.__version__, flush=True); "
     "print('CUDA runtime:', torch.version.cuda, 'CUDA available:', torch.cuda.is_available(), flush=True); "
     "assert torch.cuda.is_available(), 'Enable a GPU in Kaggle settings'; "
     "print(torch.cuda.get_device_name(0), flush=True)"],
    cwd=PROJECT_DIR, env=ENV, text=True, capture_output=True,
)
print(runtime.stdout)
if runtime.stderr:
    print(runtime.stderr)
if runtime.returncode:
    raise RuntimeError("Training runtime check failed. The underlying error is printed above.")
if RUN_TESTS:
    subprocess.run([str(PYTHON), "-m", "pytest", "-q"], cwd=PROJECT_DIR,
                   env={**ENV, "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"}, check=True)

## 3. Review repository settings before training
Epoch count, batch, image size, optimizer, lr0 and augmentations are loaded
from configs/train.yaml and native defaults. experiment.seeds selects [0,1,2]
or any requested list. GRL is disabled; output names differ from adaptive runs.
Make configuration changes in GitHub. No results path needs to be entered.

In [ ]:
baseline_command = ["scripts/run_baseline_seeds.py", "--config", TRAIN_CONFIG]
run_project([*baseline_command, "--check-only"])

## 4. Train and evaluate BASELINES ONLY

In [ ]:
run_project(baseline_command)
result = json.loads((PROJECT_DIR / "baseline_only_seeds_report.json").read_text())
BASELINE_RESULTS = Path(result["run_dir"])
print("All baseline results:", BASELINE_RESULTS)
print((BASELINE_RESULTS / "seed_summary.csv").read_text())

## 5. Review and download
BASELINE_ONLY/seed_0, seed_1, seed_2 hold separate checkpoints and evaluation
outputs. The summary contains baseline mean/sample standard deviation. A single
seed has no estimated standard deviation. The archive contains results and
copied settings, not image datasets or environments.

In [ ]:
import shutil
from IPython.display import FileLink, display
archive = shutil.make_archive("/kaggle/working/BASELINE_ONLY_Same_GRL_Seeds", "zip",
                              root_dir=BASELINE_RESULTS.parent, base_dir=BASELINE_RESULTS.name)
print("Download:", archive)
display(FileLink(archive))